# Phase 12b — prompting for clause-level precision

Moving to whole-article chunks (Phase 12a) improved retrieval and citation accuracy but cost a little on the LLM-judge scores, and the three regressions all came from the same source: an article now arrives with **all** its lettered paragraphs at once, and the generator handles that badly in three distinct ways.

| question | what went wrong | judge |
|---|---|---|
| **q36** | Cites Article 85(a) for combining pensions, then applies 85(b)'s "three times the minimum" ceiling as a general limit — that ceiling only governs work-injury disability pensioners. **Conflates two paragraphs into one rule.** | faith 5→3 |
| **q23** | Asked where social security funding comes from generally; answered entirely from Article 48, which finances *unemployment insurance* specifically. Article 19, the general provision, was in context at rank 5 and unused. **Picked a narrower provision over the governing one.** | rel 4→2 |
| **q38** | Reports Article 9's fine but omits the closure and imprisonment consequences in the sibling paragraphs of the same merged article. **Partial coverage of one article.** | rel 5→3 |

The revised prompt adds three rules aimed at exactly these: keep paragraphs distinct, cover all bearing provisions, and prefer the provision that actually governs the question.

## Round 2: rule 7 rewritten

Round 1 (results archived as `eval/prompt_precision_*_rule7v1.csv`) tested the three rules as originally drafted. Relevancy cleared the noise floor cleanly (mean 4.81→4.95). Faithfulness was a wash (4.76→4.78): q36 broke a genuine reproducible failure (both baseline runs independently scored 2, revised reached 4), but rule 7's original wording — "cover all of them... report every consequence and every exception" — told the model *what* to be complete about without constraining *how*, and on q29 it was read as license to add an unsupported interpretive claim connecting two articles (faithfulness 5→3), something neither baseline run did.

Rule 7 is rewritten to demand the same completeness without granting license to synthesize:

> 7. If more than one paragraph or article in the documents bears on the question, state what each one says, addressed separately. Do not explain how they relate to each other or why one takes precedence unless the documents say so explicitly — restate what is written, don't reason beyond it.

Rules 6 and 8 are unchanged from round 1 — no evidence against either, and they're the ones getting credit for the wins.

## Design note: a noise floor, at last

Earlier comparisons in this project (Phase 12a's A/B/C, Phase 8b's `top_k`) read differences of 1-2 questions off a **single** run per configuration, while the generator runs at default temperature and is therefore non-deterministic. That gap was flagged but never measured. This notebook fixes it by running the **current prompt twice under identical conditions**. The spread between those two runs is how much these numbers move for no reason at all — and any claim about the revised prompt has to clear it.

Retrieval is executed once per question and shared by all three arms, so the only variable is the prompt (and, between runs 1 and 2, nothing at all).

Roughly 120 generations + 120 judge calls. Kernel: `rag311`.

In [1]:
import json
import os
import re as _re
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.generation import (ARTICLE_NUM_RE, CONTEXT_SYSTEM_PROMPT, SYSTEM_PROMPT,
                            get_production_generator)
from src.retrieval import Retriever

retriever = Retriever()
eval_questions = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_draft.jsonl", encoding="utf-8")])
print(f"{len(eval_questions)} questions | {len(retriever.chunks)} article units")

40 questions | 531 article units


In [2]:
REVISED_RULES = """
6. An article is made up of separate lettered paragraphs, and each paragraph is its own rule with its own scope. State which paragraph you are relying on. Never merge conditions, rates, ceilings or exceptions from different paragraphs into a single rule, and never apply a limit stated for one category of person or benefit to a different category.
7. If more than one paragraph or article in the documents bears on the question, state what each one says, addressed separately. Do not explain how they relate to each other or why one takes precedence unless the documents say so explicitly -- restate what is written, don't reason beyond it.
8. If a provision governs only a specific sub-category (one insurance branch, one class of worker, one type of pension), say so explicitly instead of presenting it as the general rule. When both a general provision and a narrower one are present, answer from the one that actually governs the question."""

# CONTEXT_SYSTEM_PROMPT is SYSTEM_PROMPT plus a document-naming addendum; keep that addendum
# in place and slot the new rules in with the other numbered rules.
ADDENDUM = CONTEXT_SYSTEM_PROMPT[len(SYSTEM_PROMPT):]
REVISED_SYSTEM_PROMPT = SYSTEM_PROMPT + REVISED_RULES + ADDENDUM

print(REVISED_SYSTEM_PROMPT[-1400:])

ry answer with a brief disclaimer that this is general information, not legal advice, and the user should consult a licensed lawyer for their specific situation.
6. An article is made up of separate lettered paragraphs, and each paragraph is its own rule with its own scope. State which paragraph you are relying on. Never merge conditions, rates, ceilings or exceptions from different paragraphs into a single rule, and never apply a limit stated for one category of person or benefit to a different category.
7. If more than one paragraph or article in the documents bears on the question, state what each one says, addressed separately. Do not explain how they relate to each other or why one takes precedence unless the documents say so explicitly -- restate what is written, don't reason beyond it.
8. If a provision governs only a specific sub-category (one insurance branch, one class of worker, one type of pension), say so explicitly instead of presenting it as the general rule. When both a

In [3]:
# retrieve once; every arm sees identical context so the prompt is the only variable
chunks_by_qid = {q.question_id: retriever.retrieve(q.question_text)
                 for q in eval_questions.itertuples(index=False)}
print("retrieved for", len(chunks_by_qid), "questions")

retrieved for 40 questions


In [4]:
ARMS = {
    "current prompt (run 1)": None,
    "current prompt (run 2)": None,   # identical config -- this pair measures the noise floor
    "revised prompt":         REVISED_SYSTEM_PROMPT,
}

GEN_PATH = PROJECT / "eval" / "prompt_precision_generations.csv"

# Persist answers as soon as they exist. Three runs of this notebook have now died in the
# judge cell and thrown away every completed generation with them; answers are the
# expensive, non-reproducible part, so they get written before any judging starts.
if GEN_PATH.exists():
    gen_df = pd.read_csv(GEN_PATH, encoding="utf-8-sig", dtype={"question_id": str})
    print(f"loaded {len(gen_df)} cached generations from {GEN_PATH.name}")
else:
    rows = []
    for arm, prompt in ARMS.items():
        gen = get_production_generator(system_prompt=prompt)
        for q in eval_questions.itertuples(index=False):
            chunks = chunks_by_qid[q.question_id]
            res = gen.generate(q.question_text, chunks)
            cited = {a or b for a, b in ARTICLE_NUM_RE.findall(res["answer"])}
            rows.append({"arm": arm, "question_id": q.question_id,
                         "hit_correct_article": bool(cited & set(q.answer_article_numbers)),
                         "fabricated_citation": bool(cited - {c["article_number"] for c in chunks}),
                         "answer": res["answer"]})
        print("generated:", arm)
    gen_df = pd.DataFrame(rows)
    gen_df.to_csv(GEN_PATH, index=False, encoding="utf-8-sig")
    print(f"saved {len(gen_df)} generations to {GEN_PATH.name}")

answers = {(r.arm, r.question_id): r.answer for r in gen_df.itertuples(index=False)}
print(gen_df.groupby("arm")[["hit_correct_article", "fabricated_citation"]].mean().round(3).to_string())

generated: current prompt (run 1)


generated: current prompt (run 2)


generated: revised prompt
saved 120 generations to prompt_precision_generations.csv
                        hit_correct_article  fabricated_citation
arm                                                             
current prompt (run 1)                0.975                 0.05
current prompt (run 2)                0.975                 0.05
revised prompt                        0.975                 0.05


In [5]:
from openai import OpenAI

judge_client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import JUDGE_MODEL, JUDGE_PROMPT, make_judge

judge = make_judge(judge_client)
judge = make_judge(judge_client)

JUDGE_PATH = PROJECT / "eval" / "prompt_precision_judge.csv"



# Resume support: judged rows are flushed to disk as they complete, so a crash costs only
# the calls that hadn't run yet rather than the whole pass.
if JUDGE_PATH.exists():
    judge_df = pd.read_csv(JUDGE_PATH, encoding="utf-8-sig", dtype={"question_id": str})
    judge_rows = judge_df.to_dict("records")
    print(f"resuming: {len(judge_rows)} judgements already on disk")
else:
    judge_rows = []
done = {(r["arm"], r["question_id"]) for r in judge_rows}

for arm in ARMS:
    for q in eval_questions.itertuples(index=False):
        if (arm, q.question_id) in done:
            continue
        sc = judge(q.question_text, chunks_by_qid[q.question_id], answers[(arm, q.question_id)])
        judge_rows.append({"arm": arm, "question_id": q.question_id, **sc})
        pd.DataFrame(judge_rows).to_csv(JUDGE_PATH, index=False, encoding="utf-8-sig")
    print("judged:", arm)

judge_df = pd.DataFrame(judge_rows)
gen_df.merge(judge_df, on=["arm", "question_id"]).to_csv(
    PROJECT / "eval" / "prompt_precision_results.csv", index=False, encoding="utf-8-sig")

summary = (gen_df.groupby("arm")[["hit_correct_article", "fabricated_citation"]].mean()
           .join(judge_df.groupby("arm").agg(mean_faith=("faithfulness_score", "mean"),
                                             min_faith=("faithfulness_score", "min"),
                                             mean_rel=("relevancy_score", "mean"),
                                             min_rel=("relevancy_score", "min")))
           .reindex(list(ARMS)).round(3))
print(summary.to_string())

judged: current prompt (run 1)


judged: current prompt (run 2)


judged: revised prompt
                        hit_correct_article  fabricated_citation  mean_faith  min_faith  mean_rel  min_rel
arm                                                                                                       
current prompt (run 1)                0.975                 0.05       4.775          3       4.8        3
current prompt (run 2)                0.975                 0.05       4.925          3       4.8        2
revised prompt                        0.975                 0.05       4.800          3       4.8        3


## Does the revised prompt clear the noise floor?

The two "current prompt" runs differ only by the model's own sampling. Whatever gap appears between them is the amount these metrics move for no reason. The revised prompt has to beat that gap to mean anything.

In [6]:
R1, R2, REV = list(ARMS)

for metric, frame, col in [("hit_correct_article", gen_df, "hit_correct_article"),
                           ("faithfulness", judge_df, "faithfulness_score"),
                           ("relevancy", judge_df, "relevancy_score")]:
    p = frame.pivot(index="question_id", columns="arm", values=col)
    noise = (p[R1] != p[R2]).sum()
    vs1, vs2 = (p[REV] != p[R1]).sum(), (p[REV] != p[R2]).sum()
    base = (p[R1].mean() + p[R2].mean()) / 2
    print(f"{metric:20s} baseline mean {base:.3f} -> revised {p[REV].mean():.3f}")
    print(f"{'':20s} questions differing: run1 vs run2 (pure noise) = {noise};  "
          f"revised vs run1 = {vs1}, vs run2 = {vs2}")

print("\n=== the three questions this prompt targets ===")
targets = ["q36", "q23", "q38"]
jp = judge_df[judge_df.question_id.isin(targets)].pivot(index="question_id", columns="arm",
                                                        values=["faithfulness_score", "relevancy_score"])
print(jp.to_string())

hit_correct_article  baseline mean 0.975 -> revised 0.975
                     questions differing: run1 vs run2 (pure noise) = 2;  revised vs run1 = 2, vs run2 = 2
faithfulness         baseline mean 4.850 -> revised 4.800
                     questions differing: run1 vs run2 (pure noise) = 7;  revised vs run1 = 8, vs run2 = 8
relevancy            baseline mean 4.800 -> revised 4.800
                     questions differing: run1 vs run2 (pure noise) = 8;  revised vs run1 = 7, vs run2 = 8

=== the three questions this prompt targets ===
                faithfulness_score                                              relevancy_score                                      
arm         current prompt (run 1) current prompt (run 2) revised prompt current prompt (run 1) current prompt (run 2) revised prompt
question_id                                                                                                                          
q23                              5                    

In [7]:
print("=== revised-prompt answers still scoring <= 3 ===")
low = judge_df[(judge_df.arm == REV) & ((judge_df.faithfulness_score <= 3) | (judge_df.relevancy_score <= 3))]
for _, r in low.iterrows():
    print(f"{r.question_id}: faith={r.faithfulness_score} ({r.faithfulness_reason})")
    print(f"     rel={r.relevancy_score} ({r.relevancy_reason})")
if low.empty:
    print("none")

=== revised-prompt answers still scoring <= 3 ===
q17: faith=5 (The answer's core claim directly restates labor96 Article 86 without adding unsupported facts.)
     rel=3 (It correctly states the general rule but doesn't explain who is or isn't subject to the SSL (e.g., Article 4 categories), leaving the answer somewhat incomplete for practical application.)
q25: faith=3 (Mostly reproduces Article 81 accurately but introduces a distorted/garbled clause 1 (mixing sons and brothers oddly) and an odd closing line mentioning 'أصدقائك' (friends) which is not in the source at all.)
     rel=4 (Directly answers who inherits the pension upon death, listing the correct categories from Article 81, though the phrasing of point 1 is confusing and slightly inaccurate.)
q31: faith=3 (The answer correctly cites Articles 89 and 37, but conflates Article 41 (third-party injury liability of the Corporation) with Article 37's rule about gross fault of the establishment, creating a misleading blended stat